In [1]:
import numpy as np
import os
from PIL import Image

In [3]:
def load_fer(base_path, max_samples_per_class=2000):
    X = []
    y = []

    # Map target folder categories to binary SVM lables
    # Happy maps to +1, Sab maps to -1
    target_classes = {'happy': 1, 'sad': -1}

    for folder_name, label in target_classes.items():

        folder_path = os.path.join(base_path, folder_name)

        if not os.path.exists(folder_path):
            print(f"Warning: Folder not found at {folder_path}")
            continue

        print(f"Loading {folder_name} images from: {folder_path}...")
        count = 0

        for file_name in os.listdir(folder_path):

            if str(file_name).lower().endswith(('.png', '.jpg', '.jpeg')):

                file_path = os.path.join(folder_path, file_name)
                
                try:
                    # 1. Open image and explicitly force grayscale ('L' mode)
                    img = Image.open(file_path).convert('L')

                    # 2. Force resize to 48x48 to guarantee consistent input dimension
                    img = img.resize((48, 48))

                    # 3. Convert to numpy array and normalize pixels to [0.0, 1.0]
                    img_array = np.array(img, dtype=np.float32) / 255

                    # 4. Flatten the 2D (48x48) image into a 1D vector (2304 elements)
                    flat_pixels = img_array.flatten().tolist()

                    X.append(flat_pixels)
                    y.append(label)

                    count += 1

                    if count >= max_samples_per_class:
                        break


                except Exception as e:
                    pass
                
    print(f"Successfully loaded {len(X)} total samples.")
    return X, y


In [4]:
print("Your current working directory is:")
print(os.getcwd())

Your current working directory is:
/Users/tanchutat/Documents/PersonalProgramming/EmotionScan/ai-module


In [5]:
DATASET_PATH = "/Users/tanchutat/Documents/PersonalProgramming/EmotionScan/ai-module/dataset/train/"

In [6]:
X, y = load_fer(DATASET_PATH, max_samples_per_class=5000)

Loading happy images from: /Users/tanchutat/Documents/PersonalProgramming/EmotionScan/ai-module/dataset/train/happy...
Loading sad images from: /Users/tanchutat/Documents/PersonalProgramming/EmotionScan/ai-module/dataset/train/sad...
Successfully loaded 9830 total samples.


In [7]:
import csv

def save_parsed_data_to_csv(X, y, output_path="fer_processed_binary.csv"):
    print(f"Saving {len(X)} parsed samples to {output_path}...")
    
    with open(output_path, mode='w', newline='') as file:
        writer = csv.writer(file)
        
        # 1. Create a dynamic header: ['emotion', 'pixel_0', 'pixel_1', ..., 'pixel_2303']
        num_features = len(X[0])
        header = ['emotion'] + [f'pixel_{i}' for i in range(num_features)]
        writer.writerow(header)
        
        # 2. Write the label and the flattened pixel array row by row
        for pixels, label in zip(X, y):
            # Formats each pixel float to 4 decimal places to save disk space
            rounded_pixels = [round(p, 4) for p in pixels]
            row = [label] + rounded_pixels
            writer.writerow(row)
            
    print("CSV Export Complete!")

In [8]:
save_parsed_data_to_csv(X, y, "fer_processed_binary.csv")

Saving 9830 parsed samples to fer_processed_binary.csv...
CSV Export Complete!


In [9]:
def load_cached_csv(csv_path="fer_processed_binary.csv"):
    print(f"Loading cached dataset from {csv_path}...")
    X = []
    y = []
    
    with open(csv_path, mode='r') as file:
        reader = csv.reader(file)
        next(reader)  # Skip the header row
        
        for row in reader:
            # First element is the target label (+1 or -1)
            y.append(int(row[0]))
            # The remaining columns are our normalized image pixels
            X.append([float(pixel) for pixel in row[1:]])
            
    print(f"Loaded {len(X)} samples instantly.")
    return X, y

In [10]:
from main.ScratchSVM import ScratchSVM


model = ScratchSVM(learning_rate=0.001, lambda_param=0.01, n_iters=30)

In [11]:
model.fit(X, y)

In [12]:
DATA_TEST_SET_PATH = "/Users/tanchutat/Documents/PersonalProgramming/EmotionScan/ai-module/dataset/test/"

In [13]:
X_test, y_test = load_fer(DATA_TEST_SET_PATH, max_samples_per_class=5000)

Loading happy images from: /Users/tanchutat/Documents/PersonalProgramming/EmotionScan/ai-module/dataset/test/happy...
Loading sad images from: /Users/tanchutat/Documents/PersonalProgramming/EmotionScan/ai-module/dataset/test/sad...
Successfully loaded 1473 total samples.


In [14]:
save_parsed_data_to_csv(X_test, y_test, "fer_processed_binary_test.csv")

Saving 1473 parsed samples to fer_processed_binary_test.csv...
CSV Export Complete!


In [15]:
y_predicted = model.predict(X_test)

In [19]:
correct = sum(1 for true, pred in zip(y_test, y_predicted) if true == pred)
accuracy = (correct / len(y_test)) * 100
print(f"\nFinal Test Accuracy: {accuracy:.2f}%")


Final Test Accuracy: 54.04%


In [20]:
import json

def export_to_svm(model, file_name = "svm_model.json"):

    model_data = {
        "bias" : model.b,
        "weights" : [float(w_j) for w_j in model.w]
    }

    with open(file_name, 'w') as f:
        json.dump(model_data, f, indent=4)
        
    print(f"Successfully exported trained model parameters to {file_name}!")

In [21]:
export_to_svm(model)

Successfully exported trained model parameters to svm_model.json!
